In [4]:
import json
import time
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field, ValidationError, field_validator

# !pip install -q google-genai pydantic

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You are a structured healthcare operations analyst for CareLoop Clinics. "
    "Return strictly valid JSON matching the requested schema."
)

CRAFT_PROMPT_TEMPLATE = """Context: CareLoop Clinics call-centre agents phone patients who missed outpatient appointments across six Indian cities.
Role: You are a clinical operations analyst extracting structured no-show reasons from short call-centre notes (which may include English or Hinglish).
Action: Classify the note into one allowed root_cause, assign a confidence score between 0 and 1, write a concise summary of at most 25 words, and suggest one practical follow-up action. Use "other" rather than guessing when the note is ambiguous or uninformative.
Tone: Objective, concise, and factual.
Format: JSON object with keys root_cause, confidence, summary, recommended_action. Allowed root_cause values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other.

Note text:
{note_text}
"""

MODEL_NAME = "gemini-3.5-flash-lite"  # Replace with your Session 22.1 Flash model name if different
API_KEY = userdata.get("GEMINI-API-KEY")
client = genai.Client(api_key=API_KEY)

api_call_count = 0


def call_gemini(prompt_text: str, system_instruction: str = SYSTEM_TEXT):
    global api_call_count
    api_call_count += 1
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA,
    )
    return client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt_text,
        config=config,
    )


def analyse_note(note_text: str) -> dict:
    resp = call_gemini(CRAFT_PROMPT_TEMPLATE.format(note_text=note_text))
    raw_text = resp.text or ""
    parsed = json.loads(raw_text)
    return parsed, raw_text


# ---------------------------------------------------------
# Task 4.1 — Single validated call on N001
# ---------------------------------------------------------
notes_df = pd.read_csv("followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_parsed, _ = analyse_note(n001_text)
print("Task 4.1 N001 parsed dict:", n001_parsed)


# ---------------------------------------------------------
# Task 4.2 — Pydantic model & loop over first 10 notes
# ---------------------------------------------------------
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

    @field_validator("root_cause")
    @classmethod
    def validate_root_cause(cls, value: str) -> str:
        if value not in ALLOWED_ROOT_CAUSES:
            raise ValueError(f"Invalid root_cause: {value}")
        return value


rows_out = []
v2_cache = {}  # Reuse N007 and N009 results in Task 4.3(b) if desired

for _, row in notes_df.head(10).iterrows():
    note_id = row["note_id"]
    appt_id = row["appointment_id"]
    note_val = row["note_text"]

    if pd.isna(note_val) or str(note_val).strip() == "":
        rows_out.append(
            {
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": "other",
                "confidence": 0.0,
                "summary": "No note recorded",
                "recommended_action": "Re-call patient",
                "validation_status": "skipped_blank",
                "raw_response": "",
            }
        )
        continue

    raw_text = ""
    try:
        parsed_dict, raw_text = analyse_note(str(note_val))
        v2_cache[note_id] = parsed_dict.get("root_cause", "")
        validated = FollowupInsight(**parsed_dict)
        rows_out.append(
            {
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": validated.root_cause,
                "confidence": validated.confidence,
                "summary": validated.summary,
                "recommended_action": validated.recommended_action,
                "validation_status": "valid",
                "raw_response": raw_text,
            }
        )
    except Exception:
        fallback_dict = {}
        try:
            fallback_dict = json.loads(raw_text) if raw_text else {}
        except Exception:
            fallback_dict = {}
        rows_out.append(
            {
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": fallback_dict.get("root_cause", ""),
                "confidence": fallback_dict.get("confidence", np.nan),
                "summary": fallback_dict.get("summary", ""),
                "recommended_action": fallback_dict.get("recommended_action", ""),
                "validation_status": "invalid",
                "raw_response": raw_text,
            }
        )

    time.sleep(13)

notes_analysed = pd.DataFrame(rows_out)[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response",
    ]
]
notes_analysed.to_csv("notes_analysed.csv", index=False)
print("\nSaved notes_analysed.csv shape:", notes_analysed.shape)


# ---------------------------------------------------------
# Task 4.3 — Deliberately bad replies & V1 vs V2 prompt test
# ---------------------------------------------------------
bad_cases = [
    {
        "root_cause": "other",
        "confidence": 1.7,
        "summary": "Missed visit due to travel delay",
        "recommended_action": "Send reminder",
    },
    {
        "root_cause": "other",
        "confidence": 0.7,
        "summary": "Missed visit due to travel delay",
    },
]
for idx, payload in enumerate(bad_cases, 1):
    try:
        FollowupInsight(**payload)
    except ValidationError as exc:
        print(f"\nTask 4.3(a) Bad case #{idx} error:\n{exc}")

comparison_rows = []
for target_id in ["N007", "N009"]:
    note_str = str(notes_df.loc[notes_df["note_id"] == target_id, "note_text"].iloc[0])

    # V1 minimal prompt
    resp_v1 = call_gemini("Classify this note")
    try:
        rc_v1 = json.loads(resp_v1.text or "{}").get("root_cause", "")
    except Exception:
        rc_v1 = ""
    allowed_v1 = rc_v1 in ALLOWED_ROOT_CAUSES
    time.sleep(13)

    # V2 CRAFT prompt (reusing Task 4.2 result if already fetched)
    if target_id in v2_cache:
        rc_v2 = v2_cache[target_id]
    else:
        parsed_v2, _ = analyse_note(note_str)
        rc_v2 = parsed_v2.get("root_cause", "")
        time.sleep(13)
    allowed_v2 = rc_v2 in ALLOWED_ROOT_CAUSES

    comparison_rows.append(
        {
            "note_id": target_id,
            "v1_root_cause": rc_v1,
            "v1_allowed": allowed_v1,
            "v2_root_cause": rc_v2,
            "v2_allowed": allowed_v2,
        }
    )

print("\nTask 4.3(b) V1 vs V2 comparison:")
print(pd.DataFrame(comparison_rows).to_string(index=False))
print("\nTask 4.3(c) Total API calls used across 4.1-4.3:", api_call_count)


# ---------------------------------------------------------
# Task 4.4(a) — Merge & cross-tabulate root_cause vs reminder_sent
# ---------------------------------------------------------
appt_clean = pd.read_csv("appointments_clean.csv")
merged_df = notes_analysed.merge(appt_clean, on="appointment_id", how="left")
print("\nTask 4.4(a) Merged row count:", len(merged_df))
ct = pd.crosstab(merged_df["root_cause"], merged_df["reminder_sent"], margins=True)
print("\nCross-tab root_cause vs reminder_sent:\n", ct)

Task 4.1 N001 parsed dict: {'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient forgot the appointment due to missing a reminder message and requested to reschedule for next week.', 'recommended_action': 'Reschedule the appointment for next week and verify patient contact details for reminder delivery.'}

Saved notes_analysed.csv shape: (10, 8)

Task 4.3(a) Bad case #1 error:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Task 4.3(a) Bad case #2 error:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...it due to travel delay'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing

Task 4.3(b) V1 vs V2 comparison:
note_id                                    